In [1]:
import cv2
import numpy as np
from collections import deque

In [2]:
pts = deque(maxlen=512)

Lower_blue = np.array([90,50,50])
Upper_blue = np.array([150,255,255])

kernel = np.ones((5,5), np.uint8)

In [3]:
import cv2
cap = cv2.VideoCapture(0)

In [ ]:
while True:

    ret, img = cap.read()

    if not ret:
        print("Failed to grab frame")
        break

    img = cv2.flip(img, 1)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    mask = cv2.inRange(
        hsv,
        Lower_blue,
        Upper_blue
    )

    mask = cv2.erode(mask, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.dilate(mask, kernel, iterations=1)

    res = cv2.bitwise_and(img, img, mask=mask)

    contours, _ = cv2.findContours(
        mask.copy(),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    center = None

    if len(contours) > 0:

        c = max(contours, key=cv2.contourArea)

        ((x, y), radius) = cv2.minEnclosingCircle(c)

        M = cv2.moments(c)

        if M["m00"] != 0:

            center = (
                int(M["m10"] / M["m00"]),
                int(M["m01"] / M["m00"])
            )

            if radius > 5:

                cv2.circle(
                    img,
                    (int(x), int(y)),
                    int(radius),
                    (0,255,255),
                    2
                )

                cv2.circle(
                    img,
                    center,
                    5,
                    (0,0,255),
                    -1
                )

                pts.appendleft(center)

    for i in range(1, len(pts)):

        if pts[i-1] is None or pts[i] is None:
            continue

        thickness = int(
            np.sqrt(
                len(pts) / float(i + 1)
            ) * 2.5
        )

        cv2.line(
            img,
            pts[i-1],
            pts[i],
            (0,0,255),
            thickness
        )

    cv2.imshow("Frame", img)
    cv2.imshow("Mask", mask)
    cv2.imshow("Result", res)

    if cv2.waitKey(1) & 0xFF == 27:
        break

KeyboardInterrupt: 

In [ ]:
cap.release()
cv2.destroyAllWindows()